In [11]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [12]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/NGC-3217_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/NGC-3217_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     367   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        48   (2094, 964)   float32   


In [13]:
image_cut = image[67:167, 0:1890]
image_error_cut = image_error[67:167, 0:1890]

In [14]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.03191
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [15]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) #+
        #ersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [16]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/PSF/Real_seeing_NGC3217_1.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [17]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    #purple_area = single_integral(x, params6, a, b)
    #pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area# + purple_area #+ pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area]#, (purple_area*100)/sum_area]#, (pink_area*100)/sum_area]

## OIII

In [18]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit_OIII = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Fit/OIII_fit.csv", index_col = 0)

In [19]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -1, fit_OIII['Component 1'].iloc[3] + 1)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -1, fit_OIII['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -1, fit_OIII['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -1, fit_OIII['Component 4'].iloc[3] + 1)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -1, fit_OIII['Component 5'].iloc[3] + 1)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

2.1896107872064583
2.2489840927298657
2.220799735141121
2.1859662253806116
1.1270065874275992
1.127482845180594
1.0521815816121314
1.0531322419504514
0.5334639132893481
0.5303294271114579
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.765969e+01   0.951944   1.110863   0.372422   0.100095
Green      2.334678e+00  93.905034  71.869715  18.735644   7.144766
Magenta    0.000000e+00   3.713317  14.998363   0.000000   0.000000
Orange     5.033227e-10   1.086663  10.817741  76.029605   0.715881
Chocolate  5.633918e-03   0.343042   1.203319   4.862329  92.039258


In [20]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

6.2324633850692495
6.370126770228348
4.43742301707697
4.326157557347087
2.483804014251249
2.488095124679392
3.1062205785810972
3.1087112316346523
1.0693197428799666
1.0625842672609034
                 Peak 1     Peak 2     Peak 3        Peak 4     Peak 5
Blue       9.752383e+01   1.043176   1.100182  3.810088e-01   0.092895
Green      2.470173e+00  95.590535  76.480004  1.919116e+01   6.766238
Magenta    0.000000e+00   2.244296  13.644757  1.789504e-22   0.000000
Orange     8.464971e-10   0.809730   7.781348  7.544363e+01   0.504873
Chocolate  5.997366e-03   0.312263   0.993710  4.984200e+00  92.635995


In [21]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -3*sigma, fit_OIII['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

7.744101338133435
7.86888347269938
7.697746203240859
7.576220834952096
5.2728493424778735
5.280549303155087
4.010155283558908
4.014285346385677
2.0276244315100542
2.020342799187573
                 Peak 1     Peak 2     Peak 3        Peak 4     Peak 5
Blue       9.747697e+01   1.223374   1.052987  3.660223e-01   0.099076
Green      2.517139e+00  94.030171  77.846679  1.849801e+01   7.183648
Magenta    0.000000e+00   3.280157  12.112099  1.386130e-22   0.000000
Orange     6.634951e-10   1.099629   8.035003  7.542722e+01   0.673700
Chocolate  5.891711e-03   0.366669   0.953232  5.708752e+00  92.043576


## HBeta

In [22]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Fit/Hbeta_fit.csv", index_col = 0)

In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 1'].iloc[3] -1, fit_OIII['Component 1'].iloc[3] + 1)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 2'].iloc[3] -1, fit_OIII['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 3'].iloc[3] -1, fit_OIII['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 4'].iloc[3] -1, fit_OIII['Component 4'].iloc[3] + 1)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 5'].iloc[3] -1, fit_OIII['Component 5'].iloc[3] + 1)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

2.1339076025533066
2.1593942645147353
2.596634191073311
2.6298969471861495
1.4724236574522813
1.4745459016136357
0.9286764519561235
0.9293452253186896
0.6681942929549972
0.6765357621246635
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.818769e+01   0.238353   0.171918   0.033401   0.000906
Green      1.785873e+00  88.510856  61.826270  20.291742   4.069225
Magenta    0.000000e+00  10.299960  29.687952   0.000340   0.000000
Orange     1.814495e-16   0.527429   7.126798  73.754341   0.094715
Chocolate  2.643754e-02   0.423402   1.187062   5.920176  95.835154


In [24]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
#                     fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

6.145256997941559
6.204725012174791
5.059053866408768
5.072025896921133
3.2236756687275196
3.234536192970695
2.754397242102971
2.7564304771022616
1.2682924100304673
1.285966998712169
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.810217e+01   0.284934   0.183012   0.034541   0.000823
Green      1.870162e+00  91.492857  66.825563  20.729315   4.026793
Magenta    0.000000e+00   7.450791  27.048254   0.006630   0.000000
Orange     7.699092e-16   0.368178   4.939293  73.192104   0.060685
Chocolate  2.766830e-02   0.403240   1.003878   6.037410  95.911699


In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
#                     fit_OIII['Component 6'].iloc[3] -3*sigma, fit_OIII['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

7.725126855254197
7.776697863437936
8.90437175627769
8.943460475225816
6.77392234536482
6.824440744765681
3.5419629690750147
3.5454732420651025
2.3864804501243433
2.4056609164548233
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.810616e+01   0.338092   0.181966   0.031627   0.000915
Green      1.866232e+00  89.681827  68.477567  19.871313   4.319226
Magenta    0.000000e+00   8.964930  25.049959   0.005171   0.000000
Orange     5.382021e-16   0.551351   5.322585  73.201417   0.104286
Chocolate  2.760423e-02   0.463800   0.967923   6.890473  95.575573
